# Test Pasos 20-27: Tabla Final de Inversiones

> **Propósito**: Notebook limpio para probar los pasos 20-27 del modelo de inversiones.
> 
> **Prerequisitos**: Los flujos de los pasos 1-19 ya deben estar calculados.
> 
> **Fecha**: 2026-02-05

## 1. Setup y Carga de Dependencias

In [1]:
import pandas as pd
import numpy as np
import pickle
import sys
from pathlib import Path
from datetime import datetime

# Configuración de paths
notebook_path = Path().resolve()
BASE_DIR = notebook_path
while not (BASE_DIR / '.git').exists():
    BASE_DIR = BASE_DIR.parent

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

# Path a datos externos
PROCESOS_DIARIOS_PATH = BASE_DIR.parent / 'PROCESOS_DIARIOS_MODELOS'
DATA_PATH = PROCESOS_DIARIOS_PATH / 'data' / 'external' / 'ml_inversiones'

# Fecha de proceso
FECHA_PROCESO = 20260115

print(f"BASE_DIR: {BASE_DIR}")
print(f"DATA_PATH: {DATA_PATH}")
print(f"FECHA_PROCESO: {FECHA_PROCESO}")

BASE_DIR: C:\Users\vlandaetat\code\bfa-cl-modelos-diarios
DATA_PATH: C:\Users\vlandaetat\code\PROCESOS_DIARIOS_MODELOS\data\external\ml_inversiones
FECHA_PROCESO: 20260115


## 2. Importar Módulos del Proyecto

In [2]:
# Importar módulo de tabla final (pasos 20-27)
from RF_Modelo_Inversiones.output.tabla_final import (
    generar_precios_dia,
    formatear_flujo_instrumento,
    generar_tabla_final_inversiones,
    agregar_precio_y_flujo_clp,
    generar_tabla_desarrollo_completa,
    formatear_para_excel,
    ejecutar_pasos_20_a_27,
    COLUMNAS_TABLA_FINAL,
    COLUMNAS_TABLA_DESARROLLO,
    COLUMNAS_EXCEL_FINAL,
    MAPEO_COLUMNAS_EXCEL,
    extrae_cartera_ffmm,
    extraer_cartera_ffmm_para_tabla_desarrollo,
    extrae_cartera_htm,
    extraer_cartera_htm_para_tabla_desarrollo,
    extrae_cartera_rt,
    extraer_cartera_rt_para_tabla_desarrollo,
)

# Importar orquestador (para generar flujos si es necesario)
from RF_Modelo_Inversiones.pipeline.orquestador import (
    generar_flujo_liquidacion_instrumento,
    listar_tipos_instrumento
)

# Importar funciones de cartera
from RF_Modelo_Inversiones.pipeline.cartera import (
    genera_cartera_inv
)

print("✅ Módulos importados correctamente")
print(f"   - Columnas tabla final: {len(COLUMNAS_TABLA_FINAL)}")
print(f"   - Columnas tabla desarrollo: {len(COLUMNAS_TABLA_DESARROLLO)}")
print(f"   - Columnas Excel final: {len(COLUMNAS_EXCEL_FINAL)}")
print(f"   - Instrumentos disponibles: {listar_tipos_instrumento()}")

✅ Módulos importados correctamente
   - Columnas tabla final: 12
   - Columnas tabla desarrollo: 14
   - Columnas Excel final: 31
   - Instrumentos disponibles: ['GobCLP', 'GobCLF', 'DPF', 'DPR', 'BBC', 'LCH']


## 3. Cargar Datos Base (Tablas Linkeadas)

Usamos `cargar_tablas_ml_inversiones()` que:
- Abstrae la fuente de datos (PICKLE/LIVE/BIGQUERY)
- Aplica transformaciones automáticas (FPL, RF_MontosLiq)
- Permite cambiar de modo con variable de entorno o parámetro

In [3]:
# Usar el nuevo sistema de carga de datos
from RF_Modelo_Inversiones.io import (
    cargar_tablas_ml_inversiones, 
    DataSourceMode,
    listar_transformaciones
)

# Cargar tablas (modo PICKLE por defecto, con transformaciones automáticas)
tablas = cargar_tablas_ml_inversiones(
    fecha_proceso=FECHA_PROCESO,
    data_path=DATA_PATH,
    modo=DataSourceMode.PICKLE,
    verbose=True
)

print(f"\n📊 Tablas cargadas ({len(tablas)}):")
for nombre, df in tablas.items():
    print(f"   - {nombre}: {df.shape[0]:,} filas x {df.shape[1]} cols")

# Mostrar transformaciones aplicadas
print(f"\n🔧 Transformaciones registradas:")
for tabla, doc in listar_transformaciones().items():
    print(f"   - {tabla}: {doc.split(chr(10))[0]}")

📊 Cargando tablas ML Inversiones (modo: pickle)
  ✓ Cargado desde pickle: tablas_linkeadas_ml_inversiones_20260115_20260119_101906.pkl
  ✓ 12 tablas cargadas

📊 Tablas cargadas (12):
   - FPL: 7 filas x 2 cols
   - RF_base_Completa_Hist_Input: 414,802 filas x 44 cols
   - RF_Base_Diaria_Precios: 455,659 filas x 24 cols
   - RF_Base_Diaria_Precios_: 244,010 filas x 24 cols
   - RF_BD_Gestion_RM: 146,955 filas x 23 cols
   - RF_Cartera_RtaFija_Hist: 218,556 filas x 20 cols
   - RF_FactCLF_Banc: 1,350 filas x 4 cols
   - RF_FactCLF_Gob: 1,350 filas x 4 cols
   - RF_FactCLP_Banc: 1,260 filas x 4 cols
   - RF_FactCLP_Gob: 1,260 filas x 4 cols
   - RF_Fecha_Proceso_Carteras: 1 filas x 1 cols
   - RF_MontosLiq: 7 filas x 4 cols

🔧 Transformaciones registradas:
   - FPL: 
   - RF_MontosLiq: 


In [4]:
# Cargar tablas de Access (para validación)
archivos_access = list(DATA_PATH.glob(f"tablas_access_prod_{FECHA_PROCESO}_*.pkl"))
if archivos_access:
    archivo_access = max(archivos_access, key=lambda x: x.stat().st_mtime)
    print(f"📦 Cargando tablas Access: {archivo_access.name}")
    with open(archivo_access, "rb") as f:
        tablas_access = pickle.load(f)
    print(f"   Tablas disponibles: {list(tablas_access.keys())}")
else:
    print("⚠️ No hay tablas de Access para validación")
    tablas_access = {}

📦 Cargando tablas Access: tablas_access_prod_20260115_20260119_102556.pkl
   Tablas disponibles: ['Flujo_BBC', 'Flujo_DPF', 'Flujo_DPR', 'Flujo_GobCLF', 'Flujo_GobCLP', 'Flujo_LCH', 'Precios_Dia', 'RF_base_Completa_Hist', 'RF_base_Completa_Hist_30-09', 'RF_BD_Gestion_RL_30-09-2021', 'RF_CarteraBBC_Pond', 'RF_CarteraDPF_Pond', 'RF_CarteraDPR_Pond', 'RF_CarteraGobCLF_Pond', 'RF_CarteraGobCLP_Pond', 'RF_CarteraLCH_Pond', 'RF_Fecha_Proceso_Carteras_', 'RF_LI_MODELO_SALIDA_CONT_DERIV', 'RF_PLI_Modelo_Inversiones_Final_CLP', 'RF_Tabla_Desarrollo_Final', 'RF_Tabla_Desarrollo_Interna']


## 4. Preparar Tabla Base y Generar Flujos (Pasos 1-19)

Si los flujos no están pre-calculados, los generamos aquí.

In [5]:
# Importar helpers para generar RF_base_Completa_Hist
import importlib
import RF_Modelo_Inversiones.dev.helpers as helpers
importlib.reload(helpers)

# Generar tabla base histórica filtrada por fecha
tablas['RF_base_Completa_Hist'] = helpers.genera_tabla_RF_base_Completa_Hist(
    tablas['RF_base_Completa_Hist_Input'], 
    FECHA_PROCESO
)
print(f"✅ RF_base_Completa_Hist: {len(tablas['RF_base_Completa_Hist']):,} filas")

✅ RF_base_Completa_Hist: 414,802 filas


In [6]:
# Generar cartera de inversiones y pactos
tabla_fecha = pd.DataFrame({'Fecha': [pd.to_datetime(str(FECHA_PROCESO), format="%Y%m%d")]})

# Cartera disponible (pasos 1-19)
df_cartera_inv = genera_cartera_inv(
    df_base=tablas['RF_base_Completa_Hist'],
    df_fecha=tabla_fecha,
    tipo='disponible',
    verbose=True
)

# Cartera pactos (para pactos >90 días)
df_cartera_pacto = genera_cartera_inv(
    df_base=tablas['RF_base_Completa_Hist'],
    df_fecha=tabla_fecha,
    tipo='pacto',
    verbose=True
)

print(f"\n📊 Resumen:")
print(f"   - Cartera disponible: {len(df_cartera_inv):,} registros")
print(f"   - Cartera pactos: {len(df_cartera_pacto):,} registros")


GENERANDO CARTERA: Cartera Inversiones Disponible (RF_PLI_001_CarteraInv)
Registros entrada: 414,802

[JOIN] Filtro fecha proceso = 2026-01-15
  Registros que cumplen: 51,748

[WHERE] Excluir LCH: Nemotecnico[:3] <> 'LCH'
  Registros que cumplen: 408,778

[WHERE] Cod_Pro es inversión financiera
  Registros que cumplen: 11,718

[WHERE] Cod_Sub_Pro termina en: Disp, Disp_Liq, MUTUOS
  Registros que cumplen: 11,493

[WHERE] Excluir HTM: Clasificacion_Contable <> 'HTM'
  Registros que cumplen: 408,578

[WHERE FINAL] Todos los filtros combinados (AND)
  Registros que cumplen: 675

RESULTADO: 675 registros

  Distribución Instrumento:
Instrumento
BBC    356
BTP    232
BTU     68
PDB      6
DPF      6
DPR      5
OTR      2

GENERANDO CARTERA: Cartera Inversiones Pacto (RF_PLI_001d_CarteraInv_Pcto)
Registros entrada: 414,802

[JOIN] Filtro fecha proceso = 2026-01-15
  Registros que cumplen: 51,748

[WHERE] Cod_Pro es inversión financiera
  Registros que cumplen: 11,718

[WHERE] Cod_Sub_Pro te

In [7]:
# Generar flujos de liquidación para los 6 instrumentos
INSTRUMENTOS = ['GobCLP', 'GobCLF', 'DPF', 'DPR', 'BBC', 'LCH']
flujos = {}
queries_por_instrumento = {}

for instrumento in INSTRUMENTOS:
    print(f"\n{'='*60}")
    print(f"Procesando: {instrumento}")
    print('='*60)
    
    # Determinar tabla de factores
    if instrumento in ['GobCLP']:
        tabla_factores = 'RF_FactCLP_Gob'
    elif instrumento in ['GobCLF']:
        tabla_factores = 'RF_FactCLF_Gob'
    elif instrumento in ['DPF', 'BBC']:
        tabla_factores = 'RF_FactCLP_Banc'
    else:  # DPR, LCH
        tabla_factores = 'RF_FactCLF_Banc'
    
    tablas_inst = {
        tabla_factores: tablas[tabla_factores],
        'FPL': tablas['FPL'],
        'RF_MontosLiq': tablas['RF_MontosLiq']
    }
    
    flujo, queries_inst = generar_flujo_liquidacion_instrumento(
        df_cartera_inv=df_cartera_inv,
        df_cartera_inv_pacto=df_cartera_pacto,
        tablas=tablas_inst,
        tipo_instrumento=instrumento,
        fecha_proceso=FECHA_PROCESO,
        verbose=False  # Silenciar para velocidad
    )
    
    flujos[instrumento] = flujo
    queries_por_instrumento[instrumento] = queries_inst
    
    # Resumen
    monto_total = flujo['Monto_Liquidar'].sum() if 'Monto_Liquidar' in flujo.columns else 0
    print(f"   ✅ {instrumento}: {len(flujo)} días, Monto total: {monto_total:,.0f}")

print("\n" + "="*60)
print("✅ FLUJOS GENERADOS (Pasos 1-19 completados)")
print("="*60)


Procesando: GobCLP
   ✅ GobCLP: 91 días, Monto total: 1,250,345,640,998

Procesando: GobCLF
   ✅ GobCLF: 91 días, Monto total: 2,178,726

Procesando: DPF
   ✅ DPF: 91 días, Monto total: 52,122,342,432

Procesando: DPR
   ✅ DPR: 91 días, Monto total: 1,051,090

Procesando: BBC
   ✅ BBC: 91 días, Monto total: 276,303,818,362

Procesando: LCH
   ✅ LCH: 91 días, Monto total: 3,157,700

✅ FLUJOS GENERADOS (Pasos 1-19 completados)


In [8]:
DATA_PATH

WindowsPath('C:/Users/vlandaetat/code/PROCESOS_DIARIOS_MODELOS/data/external/ml_inversiones')

In [9]:
# el path al archivo de cd
df_secuencia = pd.read_csv(DATA_PATH / 'modelo_inversiones_completo_secuencia.csv',sep=';')
print(df_secuencia.loc[20]['codigo_completo'])

SELECT RF_PLI_044d_Modelo_Inversiones_Full.* INTO RF_PLI_Modelo_Inversiones_Final_CLP
FROM RF_PLI_044d_Modelo_Inversiones_Full;



In [10]:
'RF_PLI_Modelo_Inversiones_Final_CLP', #'RF_Tabla_Desarrollo_Final', 'RF_Tabla_Desarrollo_Interna'

('RF_PLI_Modelo_Inversiones_Final_CLP',)

---
# 🎯 PASOS 20-27: Tabla Final de Inversiones

A partir de aquí comenzamos con los pasos de salida.

## Paso 20: Generar Precios del Día (TCRC)

In [11]:
# Paso 20: Precios del día
df_precios_dia = generar_precios_dia(
    df_precios=tablas['RF_Base_Diaria_Precios'],
    fecha_proceso=FECHA_PROCESO,
    instrumento='TCRC',
    verbose=True
)

print(f"\n📊 Precios del día: {len(df_precios_dia)} registros")
display(df_precios_dia[df_precios_dia['NEMOTECNICO'].isin(['CLF', 'USD', 'CLP'])])


[Paso 20] Generando precios del día...
  Instrumento: TCRC
  ✓ Precios encontrados: 23 registros
    - CLF: 39,747.1200
    - USD: 883.0400

📊 Precios del día: 23 registros


,Fecha,NEMOTECNICO,Instrumento,Precio_Mid
454141,2026-01-15,CLF,TCRC,39747.12
454142,2026-01-15,CLP,TCRC,1.00
454155,2026-01-15,USD,TCRC,883.04


## Paso 21: Generar Tabla Final de Inversiones (UNION ALL)

In [12]:
# Paso 21: Tabla final de inversiones
df_tabla_final = generar_tabla_final_inversiones(
    flujos=flujos,
    fecha_proceso=FECHA_PROCESO,
    df_base=tablas['RF_base_Completa_Hist'],  # Para garantías
    df_cartera_inv_pacto=df_cartera_pacto,    # Pactos >90 días
    umbral_dias_pacto=90,
    verbose=True
)

print(f"\n📊 Tabla Final: {len(df_tabla_final):,} filas")
display(df_tabla_final.head(10))

[Paso 21] Generando tabla final de inversiones

  Procesando flujos de instrumentos:
    ✓ ML_C46_Inversiones_Financieras_GOBCLP: 6 flujos, total=623,701,831,416
    ✓ ML_C46_Inversiones_Financieras_GOBCLF: 2 flujos, total=1,084,688
    ✓ ML_C46_Inversiones_Financieras_DPFCLP: 1 flujos, total=25,992,461,906
    ✓ ML_C46_Inversiones_Financieras_DPRCLF: 5 flujos, total=523,605
    ✓ ML_C46_Inversiones_Financieras_CORPCLP: 64 flujos, total=108,127,396,636
    ✓ ML_C46_Inversiones_Financieras_CORPCLF: 4 flujos, total=1,569,972

  Generando cartera de garantías...
    ✓ Garantías: 11 registros, total=30,000,752,000

  Generando pactos fuera de plazo (>90 días)...
    ⚠ Sin pactos en cartera de entrada

  RESUMEN TABLA FINAL:
  - Total registros: 93
  - Total Cap_Amort: 787,825,620,223
  - Columnas: ['Fec_Pro', 'Cod_Emp', 'Moneda', 'Cod_A_P', 'Cod_Pro', 'Cod_Sub_Pro', 'Fec_Pago', 'Dias_Pago', 'Cap_Amort', 'Int_Total_Cont', 'VP_Cap_Amort', 'VP_Int_Total_Cont']

📊 Tabla Final: 93 filas


,Fec_Pro,Cod_Emp,Moneda,Cod_A_P,Cod_Pro,Cod_Sub_Pro,Fec_Pago,Dias_Pago,Cap_Amort,Int_Total_Cont,VP_Cap_Amort,VP_Int_Total_Cont
0,2026-01-15,1,CLP,ACT,ML_C46_Inversiones_Financieras,ML_C46_Inversiones_Financieras_GOBCLP,2026-01-16,1.0,1.110155e+11,0.0,1.110155e+11,0.0
1,2026-01-15,1,CLP,ACT,ML_C46_Inversiones_Financieras,ML_C46_Inversiones_Financieras_GOBCLP,2026-01-19,4.0,1.110155e+11,0.0,1.110155e+11,0.0
2,2026-01-15,1,CLP,ACT,ML_C46_Inversiones_Financieras,ML_C46_Inversiones_Financieras_GOBCLP,2026-01-20,5.0,1.110155e+11,0.0,1.110155e+11,0.0
3,2026-01-15,1,CLP,ACT,ML_C46_Inversiones_Financieras,ML_C46_Inversiones_Financieras_GOBCLP,2026-01-21,6.0,1.110155e+11,0.0,1.110155e+11,0.0
4,2026-01-15,1,CLP,ACT,ML_C46_Inversiones_Financieras,ML_C46_Inversiones_Financieras_GOBCLP,2026-01-22,7.0,1.110155e+11,0.0,1.110155e+11,0.0
5,2026-01-15,1,CLP,ACT,ML_C46_Inversiones_Financieras,ML_C46_Inversiones_Financieras_GOBCLP,2026-01-23,8.0,6.862433e+10,0.0,6.862433e+10,0.0
6,2026-01-15,1,CLF,ACT,ML_C46_Inversiones_Financieras,ML_C46_Inversiones_Financieras_GOBCLF,2026-01-16,1.0,7.369081e+05,0.0,7.369081e+05,0.0
7,2026-01-15,1,CLF,ACT,ML_C46_Inversiones_Financieras,ML_C46_Inversiones_Financieras_GOBCLF,2026-01-19,4.0,3.477801e+05,0.0,3.477801e+05,0.0
8,2026-01-15,1,CLP,ACT,ML_C46_Inversiones_Financieras,ML_C46_Inversiones_Financieras_DPFCLP,2026-01-16,1.0,2.599246e+10,0.0,2.599246e+10,0.0
9,2026-01-15,1,CLF,ACT,ML_C46_Inversiones_Financieras,ML_C46_Inversiones_Financieras_DPRCLF,2026-01-16,1.0,1.078761e+05,0.0,1.078761e+05,0.0


In [13]:
# las diferencias son ínfimas y seguramente son dadas por la diferencia en el trato de decimales
#  entre Python y Access, que puede redondear de forma diferente.
# Por tanto, vamos a considerar un umbral de tolerancia para considerar las celdas como iguales
# definimos un umbral de tolerancia general para considerar dos valores como iguales:
# si la diferencia entre python y Access es menor a un cierto porcentaje (en principio, 1e-6), los consideramos iguales
# es decir, si |valor_python - valor_access| / |valor_access| < 1e-6, 
# los consideramos iguales, o escrito en porcentaje, si la diferencia relativa es menor a 0.0001%, los consideramos iguales.

UMBRAL_TOLERANCIA = 1e-6

# comparamos celda a celda con la tabla de access
if 'RF_PLI_Modelo_Inversiones_Final_CLP' in tablas_access:
    df_access = tablas_access['RF_PLI_Modelo_Inversiones_Final_CLP']
    print(f"\n📊 Comparando con Access: {len(df_access)} filas")
    
    # Asegurar mismo orden de columnas
    df_tabla_final = df_tabla_final[df_access.columns]
    
    # Asegurar mismo orden de filas (por combinacion de todas las columnas en el mismo orden)
    df_tabla_final = df_tabla_final.sort_values(by=['Fec_Pro', 'Cod_Emp', 'Moneda', 'Cod_A_P', 'Cod_Pro', 'Cod_Sub_Pro',
       'Fec_Pago', 'Dias_Pago',]).reset_index(drop=True)
    df_access = df_access.sort_values(by=['Fec_Pro', 'Cod_Emp', 'Moneda', 'Cod_A_P', 'Cod_Pro', 'Cod_Sub_Pro',
       'Fec_Pago', 'Dias_Pago',]).reset_index(drop=True)

    # Comparar celda a celda
    comparacion = df_tabla_final.eq(df_access)
    total_celdas = comparacion.size
    celdas_iguales = comparacion.sum().sum()
    celdas_diferentes = total_celdas - celdas_iguales
    porcentaje_iguales = (celdas_iguales / total_celdas) * 100
    print(f"   - Total celdas: {total_celdas:,}")
    print(f"   - Celdas exactamente iguales: {celdas_iguales:,} ({porcentaje_iguales:.2f}%)")
    print(f"   - Celdas diferentes: {celdas_diferentes:,}")
    if celdas_diferentes > 0:
        print("\n   ❌ Diferencias encontradas en las siguientes filas:")
        filas_diferentes = comparacion.apply(lambda row: not row.all(), axis=1)
        # mostrar cada fila de df_tabla_final y df_access que tiene diferencias
        for idx in df_tabla_final[filas_diferentes].index:
            print(f"\n   - Fila {idx}:")
            diffs = df_tabla_final.columns[~comparacion.loc[idx]]
            for col in diffs:
                val_nuevo = df_tabla_final.at[idx, col]
                val_access = df_access.at[idx, col]

                print(f"       * Columna '{col}': Nuevo={val_nuevo} | Access={val_access}")
                # si la columna es numérica, mostrar la diferencia relativa
                if pd.api.types.is_numeric_dtype(df_tabla_final[col]):
                    if val_access != 0:
                        diff_relativa = abs(val_nuevo - val_access) / abs(val_access)
                        print(f"           - Diferencia relativa: {diff_relativa:.2e}")
                        # si la diferencia relativa es menor al umbral, considerarla como igual
                        if diff_relativa < UMBRAL_TOLERANCIA:
                            print(f"           - ✅ Dentro del umbral de tolerancia ({UMBRAL_TOLERANCIA:.2e}), se considera igual")

                        else:
                            print(f"           - ❌ Fuera del umbral de tolerancia ({UMBRAL_TOLERANCIA:.2e}), se considera diferente")
                    else:
                        print("           - Access es 0, no se puede calcular diferencia relativa")
                else:
                    print("           - No es una columna numérica, no se calcula diferencia relativa")




📊 Comparando con Access: 93 filas
   - Total celdas: 1,116
   - Celdas exactamente iguales: 1,108 (99.28%)
   - Celdas diferentes: 8

   ❌ Diferencias encontradas en las siguientes filas:

   - Fila 3:
       * Columna 'Cap_Amort': Nuevo=5117.629067447271 | Access=5117.629067447736
           - Diferencia relativa: 9.08e-14
           - ✅ Dentro del umbral de tolerancia (1.00e-06), se considera igual
       * Columna 'VP_Cap_Amort': Nuevo=5117.629067447271 | Access=5117.629067447736
           - Diferencia relativa: 9.08e-14
           - ✅ Dentro del umbral de tolerancia (1.00e-06), se considera igual

   - Fila 10:
       * Columna 'Cap_Amort': Nuevo=347780.11834022036 | Access=347780.1183402206
           - Diferencia relativa: 6.69e-16
           - ✅ Dentro del umbral de tolerancia (1.00e-06), se considera igual
       * Columna 'VP_Cap_Amort': Nuevo=347780.11834022036 | Access=347780.1183402206
           - Diferencia relativa: 6.69e-16
           - ✅ Dentro del umbral de toleranc

## Paso 23: Agregar Precio y Flujo CLP

In [14]:
# Paso 23: Agregar precio UF y calcular Flujo_CLP
# Filtrar precio CLF (UF)
df_precio_clf = df_precios_dia[df_precios_dia['NEMOTECNICO'] == 'CLF']

df_con_precio = agregar_precio_y_flujo_clp(
    df_inversiones=df_tabla_final,
    df_precios_dia=df_precio_clf,
    verbose=True
)

print(f"\n📊 Tabla con precios: {len(df_con_precio):,} filas")
print(f"   Columnas: {list(df_con_precio.columns)}")

# Mostrar ejemplo de conversión CLF -> CLP
clf_rows = df_con_precio[df_con_precio['Moneda'] == 'CLF'].head(5)
if len(clf_rows) > 0:
    print("\n🔄 Ejemplo conversión CLF → CLP:")
    display(clf_rows[['Cod_Sub_Pro', 'Moneda', 'Cap_Amort', 'Precio_Mid', 'Flujo_CLP']])

  ✓ Precio UF aplicado: 39,747.1200
  ✓ Flujo_CLP total: 944,038,406,040

📊 Tabla con precios: 93 filas
   Columnas: ['Fec_Pro', 'Cod_Emp', 'Moneda', 'Cod_A_P', 'Cod_Pro', 'Cod_Sub_Pro', 'Fec_Pago', 'Dias_Pago', 'Cap_Amort', 'Int_Total_Cont', 'VP_Cap_Amort', 'VP_Int_Total_Cont', 'Precio_Mid', 'Flujo_CLP']

🔄 Ejemplo conversión CLF → CLP:


,Cod_Sub_Pro,Moneda,Cap_Amort,Precio_Mid,Flujo_CLP
0,ML_C46_Inversiones_Financieras_CORPCLF,CLF,521618.025425,39747.12,2.073281e+10
1,ML_C46_Inversiones_Financieras_CORPCLF,CLF,521618.025425,39747.12,2.073281e+10
2,ML_C46_Inversiones_Financieras_CORPCLF,CLF,521618.025425,39747.12,2.073281e+10
3,ML_C46_Inversiones_Financieras_CORPCLF,CLF,5117.629067,39747.12,2.034110e+08
4,ML_C46_Inversiones_Financieras_DPRCLF,CLF,107876.085527,39747.12,4.287764e+09


In [15]:
FECHA_PROCESO

20260115

## Pasos 24-26: Tabla Desarrollo Completa

In [16]:
# Pasos 24-26: Generar tabla de desarrollo completa
# NOTA: FFMM, HTM, RT se extraen de cartera base (o _Input para HTM),
# usando funciones específicas que replican fielmente las queries de Access.

# --- Paso 24: FFMM ---
# Fuente: RF_base_Completa_Hist | Query: RF_PLI_044f_CarteraInv_FFMM
df_cartera_ffmm = extrae_cartera_ffmm(
    df_cartera=tablas['RF_base_Completa_Hist'],
    fecha_proceso=FECHA_PROCESO,
    verbose=True
)
# Formatear para tabla desarrollo: RF_PLI_048a_Tabla_Desarrollo_Interna_Add_FFMM
df_ffmm_desarrollo = extraer_cartera_ffmm_para_tabla_desarrollo(
    df_extrae_cartera_ffmm=df_cartera_ffmm,
    df_precios_dia=df_precio_clf,
    verbose=True
)

# --- Paso 25: HTM ---
# Fuente: RF_base_Completa_Hist_Input (¡NO _Hist!) | Query: RF_PLI_044i_CarteraInv_HTM
df_cartera_htm = extrae_cartera_htm(
    df_cartera_input=tablas['RF_base_Completa_Hist_Input'],
    fecha_proceso=FECHA_PROCESO,
    verbose=True
)
# Formatear para tabla desarrollo: RF_PLI_048b_Tabla_Desarrollo_Interna_Add_HTM
df_htm_desarrollo = extraer_cartera_htm_para_tabla_desarrollo(
    df_extrae_cartera_htm=df_cartera_htm,
    df_precios_dia=df_precio_clf,
    verbose=True
)

# --- Paso 26: RT ---
# Fuente: RF_base_Completa_Hist | Query: RF_PLI_044g_CarteraInv_RT
df_cartera_rt = extrae_cartera_rt(
    df_cartera=tablas['RF_base_Completa_Hist'],
    fecha_proceso=FECHA_PROCESO,
    verbose=True
)
# Formatear para tabla desarrollo: RF_PLI_048c_Tabla_Desarrollo_Interna_Add_RT
df_rt_desarrollo = extraer_cartera_rt_para_tabla_desarrollo(
    df_extrae_cartera_rt=df_cartera_rt,
    df_precios_dia=df_precio_clf,
    verbose=True
)

# --- Generar tabla de desarrollo completa ---
df_tabla_desarrollo = generar_tabla_desarrollo_completa(
    df_modelo_inversiones=df_tabla_final,
    df_precios_dia=df_precio_clf,
    df_cartera_ffmm=df_ffmm_desarrollo,
    df_cartera_htm=df_htm_desarrollo,
    df_cartera_rt=df_rt_desarrollo,
    verbose=True
)

print(f"\n📊 Tabla Desarrollo: {len(df_tabla_desarrollo):,} filas")


  Extrayendo cartera FFMM para fecha 2026-01-15...
    ✓ FFMM: 2 registros encontrados

  Formateando cartera FFMM para tabla de desarrollo...
  ✓ Cartera FFMM formateada para tabla de desarrollo: 2 registros

  Extrayendo cartera HTM para fecha 2026-01-15...
    ✓ HTM: 753 registros encontrados

  Formateando cartera HTM para tabla de desarrollo...
  ✓ Cartera HTM formateada para tabla de desarrollo: 753 registros

  Extrayendo cartera RT para fecha 2026-01-15...
    ⚠ Sin registros RT

  Formateando cartera RT para tabla de desarrollo...
  ✓ Cartera RT formateada para tabla de desarrollo: 0 registros

[Pasos 22-26] Generando tabla de desarrollo completa

  [23] Procesando modelo de liquidación...
  ✓ Precio UF aplicado: 39,747.1200
  ✓ Flujo_CLP total: 944,038,406,040
       ML: 93 registros

  [24] Agregando FFMM (ya formateado)...
       FFMM: 2 registros

  [25] Agregando HTM (ya formateado)...
       HTM: 753 registros

  RESUMEN TABLA DESARROLLO:
  - Total registros: 848
  - To

## Paso 27: Formatear para Excel

In [17]:
# Paso 27: Formatear para Excel (RF_PLI_049 → RF_PLI_050)
# NOTA: Usa las carteras completas (extrae_cartera_*), NO las _para_tabla_desarrollo,
# porque necesita columnas como Cod_Estrategia, Clasificacion_Contable, Fec_Vcto, etc.
df_excel = formatear_para_excel(
    df_tabla_final=df_tabla_final,
    df_cartera_ffmm=df_cartera_ffmm,
    df_cartera_htm=df_cartera_htm,
    df_cartera_rt=df_cartera_rt,
    verbose=True
)

print(f"\n📊 Formato Excel: {len(df_excel):,} filas x {len(df_excel.columns)} columnas")
print(f"   Columnas: {list(df_excel.columns)}")
display(df_excel.head())


[Paso 27] Formateando para Excel (RF_PLI_049 → RF_PLI_050)...
  ML:   93 registros
  FFMM: 2 registros
  HTM:  753 registros
  ✓ Formateado: 848 registros, 31 columnas

📊 Formato Excel: 848 filas x 31 columnas
   Columnas: ['FECHA PROCESO', 'CODIGO_EMPRESA', 'OPERACION', 'COD ACT/PAS', 'MONEDA_ORIGEN', 'MONEDA_COMPENSACION', 'COMPENSACION', 'CODIGO_PRODUCTO', 'CODIGO_SUBPRODUCTO', 'FECHA CREACION', 'NUMERO_CUOTA', 'FECHA_INICIO_CUOTA', 'FECHA_VENCIMIENTO_CUOTA', 'FECHA PAGO', 'FECHA_REPRICING', 'AMORTIZACION', 'INTERES', 'INTERES_DEVENGADO', 'VP_AMORTIZACION', 'VP_INTERES', 'FACTOR DE RIESGO', 'TIPO_CUOTA', 'AREA NEGOCIO', 'CODIGO_ EJECUTIVO', 'CODIGO_ESTRATEGIA', 'CLASIFICACION_CONTABLE', 'TIPO TASA', 'INDEXADOR', 'TASA', 'TASA CF', 'SPREAD']


,FECHA PROCESO,CODIGO_EMPRESA,OPERACION,COD ACT/PAS,MONEDA_ORIGEN,MONEDA_COMPENSACION,COMPENSACION,CODIGO_PRODUCTO,CODIGO_SUBPRODUCTO,FECHA CREACION,...,TIPO_CUOTA,AREA NEGOCIO,CODIGO_ EJECUTIVO,CODIGO_ESTRATEGIA,CLASIFICACION_CONTABLE,TIPO TASA,INDEXADOR,TASA,TASA CF,SPREAD
0,2026-01-15,1.0,,ACT,CLF,CLF,C,ML_C46_Inversiones_Financieras,ML_C46_Inversiones_Financieras_CORPCLF,,...,1,TRADING TASAS,,TRADING TASAS,P&L,1,,,,
1,2026-01-15,1.0,,ACT,CLF,CLF,C,ML_C46_Inversiones_Financieras,ML_C46_Inversiones_Financieras_CORPCLF,,...,1,TRADING TASAS,,TRADING TASAS,P&L,1,,,,
2,2026-01-15,1.0,,ACT,CLF,CLF,C,ML_C46_Inversiones_Financieras,ML_C46_Inversiones_Financieras_CORPCLF,,...,1,TRADING TASAS,,TRADING TASAS,P&L,1,,,,
3,2026-01-15,1.0,,ACT,CLF,CLF,C,ML_C46_Inversiones_Financieras,ML_C46_Inversiones_Financieras_CORPCLF,,...,1,TRADING TASAS,,TRADING TASAS,P&L,1,,,,
4,2026-01-15,1.0,,ACT,CLF,CLF,C,ML_C46_Inversiones_Financieras,ML_C46_Inversiones_Financieras_DPRCLF,,...,1,TRADING TASAS,,TRADING TASAS,P&L,1,,,,


---
## 🧪 Validación vs Access

In [ ]:
# Comparar flujos generados vs Access
flujos_access = {}
for inst in ['GobCLP', 'GobCLF', 'DPF', 'DPR', 'BBC', 'LCH']:
    key = f'Flujo_{inst}'
    if key in tablas_access:
        flujos_access[inst] = tablas_access[key]

if flujos_access:
    print("="*70)
    print("COMPARACIÓN FLUJOS: Python vs Access")
    print("="*70)
    
    for inst in flujos.keys():
        if inst not in flujos_access:
            continue
            
        df_py = flujos[inst]
        df_acc = flujos_access[inst]
        
        # Comparar totales
        total_py = df_py['Monto_Liquidar'].sum() if 'Monto_Liquidar' in df_py.columns else 0
        total_acc = df_acc['Monto_Liquidar'].sum() if 'Monto_Liquidar' in df_acc.columns else 0
        diff = abs(total_py - total_acc)
        
        status = "✅" if diff < 1 else "⚠️"
        print(f"\n{inst}:")
        print(f"   Python: {total_py:20,.2f}")
        print(f"   Access: {total_acc:20,.2f}")
        print(f"   Diff:   {diff:20,.2f} {status}")
else:
    print("⚠️ No hay datos de Access disponibles para validación")

---
## 📦 Función Wrapper Completa

In [ ]:
# Ejecutar todo de una vez con la función wrapper
tablas_input = {
    'RF_Base_Diaria_Precios': tablas['RF_Base_Diaria_Precios'],
    'RF_base_Completa_Hist': tablas['RF_base_Completa_Hist']
}

resultado = ejecutar_pasos_20_a_27(
    flujos=flujos,
    tablas=tablas_input,
    fecha_proceso=FECHA_PROCESO,
    df_cartera_inv_pacto=df_cartera_pacto,
    verbose=True
)

print("\n" + "="*60)
print("📦 RESULTADO FINAL")
print("="*60)
print(f"✅ Precios día:      {len(resultado['precios_dia']):,} filas")
print(f"✅ Tabla final:      {len(resultado['tabla_final_inversiones']):,} filas")
print(f"✅ Tabla desarrollo: {len(resultado['tabla_desarrollo']):,} filas")
print(f"✅ Formato Excel:    {len(resultado['tabla_excel']):,} filas")

---
## 💾 Exportar a Excel (opcional)

In [ ]:
# Exportar a Excel si es necesario
EXPORTAR = False  # Cambiar a True para exportar

if EXPORTAR:
    output_path = DATA_PATH / f"ML_Inversiones_Output_{FECHA_PROCESO}.xlsx"
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        resultado['tabla_excel'].to_excel(writer, sheet_name='Tabla_Desarrollo', index=False)
        resultado['tabla_final_inversiones'].to_excel(writer, sheet_name='Tabla_Final', index=False)
    print(f"✅ Exportado a: {output_path}")
else:
    print("ℹ️ Exportación deshabilitada. Cambiar EXPORTAR=True para exportar.")